In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "requirements.txt").is_file() and (p / "library/nju-llm-2026").is_dir()
)
MODEL_DIR = PROJECT_ROOT / ".cache/llm-course/qwen3-tokenizer"
if not (MODEL_DIR / "model.safetensors").is_file():
    raise FileNotFoundError(f"模型尚未完整下载：{MODEL_DIR}；请查看 library/nju-llm-2026/README.md")
os.environ["LECTURE4_MODEL_PATH"] = str(MODEL_DIR)
os.environ["LECTURE4_TOKENIZER_PATH"] = str(MODEL_DIR)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["USE_TF"] = "0"
print("本地模型：", MODEL_DIR)


In [ ]:
import torch

def main():
    print("Hello from torch-test!")
    if torch.backends.mps.is_available():
        print("Excellent! MPS backend is available.")
    else:
        print("MPS backend is not available: Something went wrong! Are you running this on a Mac with Apple Silicon chip?")

if __name__ == "__main__":
    main()

In [ ]:
import transformers
print(transformers.__version__)
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
model_name = str(MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
print(tokenizer)

In [ ]:
# load the tokenizer and the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    local_files_only=True,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # 首次学习使用短回答；需要时可改为 True。
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
print(prompt)
print(text)
print(model_inputs)


In [ ]:
# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=64
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

In [ ]:
# conduct text completion with streaming output
from transformers import TextStreamer

# Create a streamer for real-time output
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# Generate with streaming
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=64,
    streamer=streamer,  # Enable streaming
    do_sample=True,
    temperature=0.7
)